# AI Escape Room — run in Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elifsila/AI-Escape-Room/blob/main/run_in_colab.ipynb)

This notebook clones the repository, installs the dependencies and starts the Streamlit apps.
Colab cannot show a Streamlit app directly, so the app is opened through a free temporary
Cloudflare tunnel link (`https://....trycloudflare.com`).

**Setup (once)**
1. Get a free Gemini API key at https://aistudio.google.com/apikey
2. In Colab, open **Secrets** (key icon on the left), add `GEMINI_API_KEY` with your key and turn on *Notebook access*.

Then run the cells from top to bottom. Version 1 and Version 2 each get their own link.

In [ ]:
# 1) Get the code (clone the first time, pull the latest changes afterwards)
import os
REPO_URL = "https://github.com/elifsila/AI-Escape-Room.git"
if os.path.exists("/content/AI-Escape-Room"):
    %cd /content/AI-Escape-Room
    !git pull
else:
    !git clone {REPO_URL} /content/AI-Escape-Room
    %cd /content/AI-Escape-Room
!pip install -q -r requirements.txt

In [ ]:
# 2) Create the .env file from Colab Secrets (the key is not printed and .env is git-ignored)
# If the model returns 429/503, change GEMINI_MODEL (e.g. gemini-3.6-flash).
from google.colab import userdata
key = userdata.get("GEMINI_API_KEY")
with open(".env", "w") as f:
    f.write(f"GEMINI_API_KEY={key}\nGEMINI_MODEL=gemini-3.5-flash-lite\nOFFLINE_MODE=0\n")
print(".env created")

In [ ]:
# 3) Run the Version 2 tests (offline, no API calls)
!python -m pytest -v

In [ ]:
# 4) Download the Cloudflare tunnel and define a launcher
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
import subprocess, time, re

def launch(app):
    """Stop any running app, start `app` with Streamlit and print a public link."""
    subprocess.run('pkill -f "streamlit run"; pkill -f cloudflared', shell=True)
    subprocess.Popen(f"streamlit run {app} --server.port 8501 --server.headless true > streamlit.log 2>&1", shell=True)
    time.sleep(5)
    subprocess.Popen("./cloudflared tunnel --url http://localhost:8501 > tunnel.log 2>&1", shell=True)
    for _ in range(40):
        time.sleep(1)
        m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("tunnel.log").read())
        if m:
            print("Open this link:", m.group(0))
            return
    print("No link yet. Last log lines:")
    print(open("tunnel.log").read()[-1500:])

## Version 1 — prompt-only Game Master
If the page shows an error, check `streamlit.log` in the Files panel.

In [ ]:
launch("v1/app.py")

## Version 2 — game engine + LLM narrator
This stops Version 1 and prints a new link.

In [ ]:
launch("v2/app.py")

## Stop the apps

In [ ]:
!pkill -f "streamlit run"; pkill -f cloudflared